# Primer3 Thermodynamics

Score DNA oligos for the thermodynamic properties that decide whether a primer works: melting temperature (Tm), hairpin and dimer stability (ΔG), GC content, and a 3' GC-clamp. Pair a forward primer with its reverse as a `partner` to check the pair for cross-dimerization.

This wraps [primer3-py](https://github.com/libnano/primer3-py), the Cython binding to [Primer3](https://primer3.org/) ([Untergasser et al., 2012](https://doi.org/10.1093/nar/gks596)).

This example is oriented toward **qPCR** primer design, but the metrics apply to PCR and sequencing primers generally.

In [1]:
from proto_tools.tools.sequence_scoring.primer3 import (
    Primer3ThermodynamicsConfig,
    Primer3ThermodynamicsInput,
    run_primer3_thermodynamics,
)
from proto_tools.utils.notebook_docs import display_api_reference

## API reference

In [2]:
display_api_reference("primer3-thermodynamics", "input", "run_primer3_thermodynamics")

**Input** — `Primer3ThermodynamicsInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>oligos</code> | <code>list[Primer3Oligo]</code> | required | DNA oligos to score (each optionally paired with a partner for heterodimer ΔG) |

In [3]:
display_api_reference("primer3-thermodynamics", "config", "run_primer3_thermodynamics")

**Config** — `Primer3ThermodynamicsConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>mv_conc</code> | <code>float</code> | <code>50.0</code> | Monovalent cation concentration in mM (e.g. Na+, K+) |
| <code>dv_conc</code> | <code>float</code> | <code>1.5</code> | Divalent cation concentration in mM (Mg2+); raises Tm and dimer stability |
| <code>dntp_conc</code> | <code>float</code> | <code>0.6</code> | dNTP concentration in mM; sequesters Mg2+, lowering effective divalent |
| <code>dna_conc</code> | <code>float</code> | <code>50.0</code> | Oligo/DNA concentration in nM; affects Tm and dimer ΔG |
| <code>temp_c</code> | <code>float</code> | <code>37.0</code> | Temperature in °C at which hairpin/homodimer/heterodimer ΔG is evaluated |

In [4]:
display_api_reference("primer3-thermodynamics", "output", "run_primer3_thermodynamics")

**Output** — `Primer3ThermodynamicsOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[Primer3OligoResult]</code> | <code>[]</code> | Per-oligo thermodynamic scores |

**`Primer3OligoResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>oligo_id</code> | <code>str</code> | required | Positional label of the input oligo |
| <code>length</code> | <code>int</code> | required | Oligo length in nucleotides |
| <code>tm</code> | <code>float</code> | required | Melting temperature in °C |
| <code>hairpin_dg</code> | <code>float</code> | required | Hairpin ΔG in kcal/mol |
| <code>homodimer_dg</code> | <code>float</code> | required | Self-dimer ΔG in kcal/mol |
| <code>heterodimer_dg</code> | <code>float &#124; None</code> | <code>None</code> | Cross-dimer ΔG with the partner in kcal/mol |
| <code>gc_content</code> | <code>float</code> | required | Fraction of G/C bases (0-1) |
| <code>gc_clamp</code> | <code>bool</code> | required | True if a G/C is in the last two 3' bases |
| <code>hairpin_structure_found</code> | <code>bool</code> | required | Whether a hairpin structure was found |
| <code>homodimer_structure_found</code> | <code>bool</code> | required | Whether a homodimer structure was found |
| <code>heterodimer_structure_found</code> | <code>bool &#124; None</code> | <code>None</code> | Whether a heterodimer structure was found (None if no partner) |

## Basic usage: score a qPCR primer pair

A GAPDH-style forward/reverse pair. We pass the reverse primer as the forward's `partner`, so the heterodimer ΔG between the two is computed.

In [5]:
fwd = "ACCCACTCCTCCACCTTTGA"
rev = "CTGTTGCTGTAGCCAAATTCGT"

result = run_primer3_thermodynamics(
    Primer3ThermodynamicsInput(oligos=[{"sequence": fwd, "partner": rev}]),
    Primer3ThermodynamicsConfig(),
)

r = result.results[0]
print(f"Tm:             {r.tm:.1f} °C   (qPCR target 58-62 °C)")
print(f"GC content:     {r.gc_content:.0%}      (target 40-60%)")
print(f"GC clamp:       {r.gc_clamp}")
print(f"Hairpin ΔG:     {r.hairpin_dg:.2f} kcal/mol   (want > -2)")
print(f"Homodimer ΔG:   {r.homodimer_dg:.2f} kcal/mol   (want > -6)")
print(f"Heterodimer ΔG: {r.heterodimer_dg:.2f} kcal/mol   (fwd x rev; want > -6)")

Running run_primer3_thermodynamics [00:00]

Tm:             60.4 °C   (qPCR target 58-62 °C)
GC content:     55%      (target 40-60%)
GC clamp:       True
Hairpin ΔG:     0.00 kcal/mol   (want > -2)
Homodimer ΔG:   -0.98 kcal/mol   (want > -6)
Heterodimer ΔG: -2.57 kcal/mol   (fwd x rev; want > -6)


## Advanced usage: batch scoring under qPCR conditions

`primer3-py`'s defaults reproduce Primer3 directly; they are not a qPCR preset. Set the ionic/oligo conditions to match your master mix (here: higher Mg2+, dNTP, and oligo concentration). Multiple oligos are scored in one call and returned in input order.

In [6]:
qpcr_config = Primer3ThermodynamicsConfig(
    dv_conc=3.0,      # Mg2+ (mM)
    dntp_conc=0.8,    # dNTP (mM)
    dna_conc=250.0,   # oligo (nM)
    temp_c=60.0,      # evaluate hairpin/dimer ΔG near the annealing temperature
)

candidates = [
    "ACCCACTCCTCCACCTTTGA",
    "GTGGTGAAGCAGGCATCTGA",
    "GCGCGCAAAAAGCGCGC",      # self-complementary: expect a strong hairpin
]

batch = run_primer3_thermodynamics(
    Primer3ThermodynamicsInput(oligos=candidates),
    qpcr_config,
)

print(f"{'oligo':22} {'Tm':>6} {'GC':>5} {'clamp':>6} {'hairpin':>8} {'homodimer':>10}")
for seq, res in zip(candidates, batch.results):
    print(f"{seq:22} {res.tm:6.1f} {res.gc_content:5.0%} {str(res.gc_clamp):>6} {res.hairpin_dg:8.2f} {res.homodimer_dg:10.2f}")

Running run_primer3_thermodynamics [00:00]

oligo                      Tm    GC  clamp  hairpin  homodimer
ACCCACTCCTCCACCTTTGA     64.5   55%   True     0.00      -0.09
GTGGTGAAGCAGGCATCTGA     64.3   55%   True     0.00       1.40
GCGCGCAAAAAGCGCGC        69.2   71%   True    -4.56      -8.30


## Export results

In [7]:
from pathlib import Path

out_dir = Path("./primer3_results")
out_dir.mkdir(exist_ok=True)
batch.export(name="primers", export_path=str(out_dir), file_format="csv")
batch.export(name="primers", export_path=str(out_dir), file_format="json")

print("Wrote:", sorted(p.name for p in out_dir.iterdir()))

Wrote: ['primers.csv', 'primers.json']
